In [5]:
import csv
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit import IndicProcessor

CKPT = "ai4bharat/indictrans2-indic-en-dist-200M"
SRC_LANG = "san_Deva"
TGT_LANG = "eng_Latn"
MAX_LEN = 256
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

ip = IndicProcessor(inference=True)
tokenizer = AutoTokenizer.from_pretrained(CKPT, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(CKPT, trust_remote_code=True).to(DEVICE)
model.eval()
print("Model loaded.")

Device: cuda
Model loaded.


In [6]:
with open("../data/ncert_sentences.txt", encoding="utf-8") as f:
    sentences = [line.strip() for line in f if line.strip()]

print(f"Loaded {len(sentences)} sentences")

batch = ip.preprocess_batch(sentences, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
inputs = tokenizer(batch, padding="longest", truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)

with torch.inference_mode():
    generated = model.generate(**inputs, max_length=MAX_LEN, num_beams=5)

decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
translations = ip.postprocess_batch(decoded, lang=TGT_LANG)

for src, out in zip(sentences, translations):
    print(f"{src}\n  → {out}\n")

Loaded 11 sentences
आसीत् शोभावती नाम काचन नगरी।
  → It was a small town called Shobhavati.

तत्र शूद्रको नाम महापराक्रमी नानाशास्त्रवित् पूतचरित्रः महीपतिः प्रतिवसति स्म।
  → It was inhabited by Shudrako, a great scholar, and Mahipati, a great scholar.

अथैकदा वीरवरनामा राजपुत्रः वृत्त्यर्थं कस्मादपि देशाद् राजद्वारमुपागच्छत्।
  → Once upon a time, a prince named Veeravarnama went to the palace from any country for a profession.

तेन सह वेदरता नाम तस्य पत्नी, शक्तिधरो नाम सुतः, वीरवती नाम कन्या च समायाताः।
  → He was accompanied by his wife named Vedarvati, his son named Shakti Dharo, and his daughter named Veeravati.

भोः प्रतिहार! वृत्त्यर्थमागतो राजपुत्रोऽस्मि, तस्मान्नय मां स्वामिनः समीपम्।
  → Behold, I have come back to the kingdom, so I must go back to my master.

देव! यदि सौभाग्येन अहं भवतः सेवायां नियोजितः तर्हि यदादिश्यते तत् श्रद्धया पालयिष्यामि।
  → God, if I am lucky enough to be of service to you, I will treat it with reverence if I see it.

किं ते वर्तनम्?
  → Are they 

In [7]:
import os
os.makedirs("../outputs", exist_ok=True)

with open("../outputs/test_out.csv", "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["sanskrit", "model_output", "reference_english", "notes"])
    for src, out in zip(sentences, translations):
        writer.writerow([src, out, "", ""])
print("Saved to ../outputs/test_out.csv")

Saved to ../outputs/test_out.csv


In [8]:
sentences = [
    "बालकः विद्यालयं गच्छति।",
    "सा पुस्तकं पठति।",
    "अहं जलं पिबामि।",
    "वृक्षे पक्षिणः वसन्ति।",
    "राजा नगरं पालयति।",
]

batch = ip.preprocess_batch(sentences, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
inputs = tokenizer(batch, padding="longest", truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)

with torch.inference_mode():
    generated = model.generate(**inputs, max_length=MAX_LEN, num_beams=5)

decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
translations = ip.postprocess_batch(decoded, lang=TGT_LANG)

for src, out in zip(sentences, translations):
    print(f"{src}\n  → {out}\n")

बालकः विद्यालयं गच्छति।
  → The child goes to school.

सा पुस्तकं पठति।
  → She reads books.

अहं जलं पिबामि।
  → I will drink the water.

वृक्षे पक्षिणः वसन्ति।
  → Birds live in trees.

राजा नगरं पालयति।
  → The king guards the city.



In [12]:
with open("../outputs/baseline_output.csv", "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["sanskrit", "model_output", "reference_english", "notes"])
    for src, out in zip(sentences, translations):
        writer.writerow([src, out, "", ""])
print("Saved to ../outputs/baseline_output.csv")

Saved to ../outputs/baseline_output.csv


In [17]:
# from sacrebleu.metrics import CHRF, BLEU
# import pandas as pd
# df = pd.read_csv("../outputs/baseline_output.csv")
# hyps, refs = df["model_output"].tolist(), df["reference_english"].tolist()
# print("chrF:", CHRF().corpus_score(hyps, [refs]).score)
# print("BLEU:", BLEU().corpus_score(hyps, [refs]).score)

In [15]:
import pandas as pd
df = pd.read_csv("../outputs/baseline_output.csv")
print(df["reference_english"].isna().sum(), "of", len(df), "references are empty")

5 of 5 references are empty


In [18]:
# from sacrebleu.metrics import CHRF, BLEU
# import pandas as pd

# df = pd.read_csv("../outputs/baseline_output.csv")
# df = df.dropna(subset=["reference_english"])      # skip rows with no reference
# print(len(df), "rows scored")

# hyps = df["model_output"].astype(str).tolist()
# refs = df["reference_english"].astype(str).tolist()
# print("chrF:", CHRF().corpus_score(hyps, [refs]).score)
# print("BLEU:", BLEU().corpus_score(hyps, [refs]).score)